# Solar filament segmentation: the full pipeline

This notebook walks through the solution end to end: the data and its
multi-annotator ground truth, the cross-validation split, training of the 12
models, inference on the test set, validation of the submission, an exact-PQ
evaluation on fold 0 and a look at the predictions.

```
2048 x 2048 H-alpha frame
  -> 7 YOLO11 voters (boxes)  -> union-find clusters, keep if >= 5 voters at conf >= 0.40
  -> 5 U-Net refiners x 3 flipped views on the seed box  -> pixel majorities
  -> greedy non-overlapping painting + area / contrast gates  -> submission.csv
```

Heavy steps are switched by the flags in the next cell. With the defaults and
no GPU, the notebook runs the data exploration, the split, the metric
illustration and skips everything that needs trained weights or a GPU.

In [ ]:
import glob
import os
import sys
import time
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

RUN_TRAINING = False        # trains all 12 models (~9-10 GPU hours on a P100)
RUN_INFERENCE = True        # test-set submission (~25 min on a T4); skipped without weights
RUN_FOLD0_EVAL = True       # fold-0 single-model PQ; skipped without weights
FOLD0_IMAGES = None         # None = the whole fold; a small number gives a quick, noisy look on CPU
WEIGHTS = REPO / "weights"

# On Kaggle the competition data is mounted under /kaggle/input
if "MAGFILO_DIR" not in os.environ:
    hits = glob.glob("/kaggle/input/**/MAGFiLO_1.0_Kaggle_2026", recursive=True)
    if hits:
        os.environ["MAGFILO_DIR"] = hits[0]

## 1. Environment check

Versions matter for exact reproduction: the submitted predictions were made
with ultralytics 8.4.173 and segmentation-models-pytorch 0.5.0 (see
`requirements.txt`).

In [ ]:
import importlib.metadata as md

import torch

from filament.data import TEST_IMAGES, TRAIN_IMAGES, data_root

print("python", sys.version.split()[0])
for pkg in ["torch", "torchvision", "ultralytics", "segmentation-models-pytorch", "timm",
            "pycocotools", "opencv-python", "opencv-python-headless", "numpy", "pandas"]:
    try:
        print(f"{pkg:30s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:30s} -")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", torch.cuda.get_device_name() if DEVICE.type == "cuda" else "cpu")
print("data root:", data_root(), "exists:", data_root().exists())

## 2. Data: images, annotator records, filaments

The ground truth is organised by **record**: one annotator's complete
labelling of one image. Many images were labelled two or three times, and the
metric scores predictions against every record separately.

In [ ]:
from collections import Counter

import numpy as np

from filament.data import load_records

records = load_records()
stems = sorted({r.stem for r in records})
records_per_image = Counter(Counter(r.stem for r in records).values())
areas = np.array([f.area for r in records for f in r.filaments])
short_side = np.array([min(f.bbox[2], f.bbox[3]) for r in records for f in r.filaments])

print(f"{len(stems)} training images, {len(records)} records, {len(areas)} filaments")
print("records per image:", dict(sorted(records_per_image.items())))
print("filaments per record: median", int(np.median([len(r.filaments) for r in records])))
print(f"filament area: median {np.median(areas):.0f} px, 10-90 % range "
      f"{np.percentile(areas, 10):.0f}-{np.percentile(areas, 90):.0f} px of a 2048 x 2048 frame")
print("chirality categories:", dict(sorted(Counter(f.category_id for r in records for f in r.filaments).items())))
print("test images:", len(list((data_root() / TEST_IMAGES).glob("*.jpeg"))))

### Annotator disagreement

Below, one image labelled by three annotators: each colour is one record.
Annotators mostly agree on large, dark filaments and disagree on faint or
fragmented ones, and on where exactly a ribbon ends. The zoom shows how thin
the targets are at native resolution, which is why the images are never
downsampled for mask prediction.

In [ ]:
import matplotlib.pyplot as plt

from filament.data import load_image_gray

by_stem = {}
for r in records:
    by_stem.setdefault(r.stem, []).append(r)
stem = max((s for s in by_stem if len(by_stem[s]) == 3), key=lambda s: sum(len(r.filaments) for r in by_stem[s]))
img = load_image_gray(data_root() / TRAIN_IMAGES, stem + ".jpeg")
colors = ["#e6194b", "#3cb44b", "#4363d8"]

# zoom on the region where the first record's largest filament lies
big = max(by_stem[stem][0].filaments, key=lambda f: f.area)
cx, cy = big.bbox[0] + big.bbox[2] / 2, big.bbox[1] + big.bbox[3] / 2
x0, y0 = int(np.clip(cx - 200, 0, 1648)), int(np.clip(cy - 200, 0, 1648))

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, (xa, xb, ya, yb) in zip(axes, [(0, 2048, 0, 2048), (x0, x0 + 400, y0, y0 + 400)]):
    ax.imshow(img, cmap="gray")
    for color, rec in zip(colors, by_stem[stem]):
        for f in rec.filaments:
            poly = np.array(f.polygon).reshape(-1, 2)
            ax.plot(*np.vstack([poly, poly[:1]]).T, color=color, lw=0.8)
        ax.plot([], [], color=color, label=f"record {rec.record_id.split('-')[0]}: {len(rec.filaments)} filaments")
    ax.set_xlim(xa, xb)
    ax.set_ylim(yb, ya)
    ax.axis("off")
axes[0].legend(loc="lower left", fontsize=9)
axes[0].set_title(stem)
axes[1].set_title("400 x 400 px zoom")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 3))
plt.hist(areas, bins=np.logspace(1, 5.5, 60), color="#4363d8")
plt.xscale("log")
plt.xlabel("filament area (px)")
plt.ylabel("count")
plt.title("Filament areas: tiny objects on a 4.2 Mpx frame")
plt.show()

## 3. Cross-validation split

Folds group images by **observing day**: frames taken minutes apart look
nearly identical, so any finer split leaks pixels between training and
validation. The committed `data/cv_folds.json` regenerates exactly from the
annotations.

In [ ]:
import json

from filament.data import FOLDS_PATH, split_stems

folds = split_stems(records, n_folds=5, seed=0)
committed = json.loads(FOLDS_PATH.read_text())["folds"]
assert [sorted(f) for f in folds] == committed, "fold split does not regenerate"
day_fold = {}
for k, fold in enumerate(folds):
    for s in fold:
        assert day_fold.setdefault(s[:8], k) == k, "a day straddles two folds"
for k, fold in enumerate(folds):
    recs = [r for r in records if r.stem in fold]
    print(f"fold {k}: {len(fold)} images, {len({s[:8] for s in fold})} days, {len(recs)} records, "
          f"{sum(len(r.filaments) for r in recs)} filaments")

## 4. The metric

`filament.metric` reproduces the organisers' PQ exactly: TP IoUs, FP and FN
are pooled over the whole set; ground truth is per record; every pair with
IoU > 0.5 is a TP (no one-to-one matching).

A useful reference point: score one annotator as if it were a submission
against the *other* annotators of the same images. On the 296 multi-record
training images this gives a PQ of about 0.34, below the model's out-of-fold
PQ of about 0.41: much of the remaining error is annotation ambiguity, and a
model trained on all records learns something closer to their consensus.

In [ ]:
from filament.data import gt_rows_for
from filament.metric import pq_from_rles

multi = [s for s in by_stem if len(by_stem[s]) > 1]
first = [by_stem[s][0] for s in multi]
others = [r for s in multi for r in by_stem[s][1:]]
human_pred = [(f"{r.stem}_{k}", rle) for r in first for k, (_, rle) in enumerate(gt_rows_for([r]), 1)]
res = pq_from_rles(gt_rows_for(others), human_pred)
print(f"one annotator vs the others on {len(multi)} multi-record images:",
      {k: round(v, 3) if isinstance(v, float) else v for k, v in res.items()})

## 5. Training (12 models)

* 7 detectors: YOLO11s at 1280 px for each of the 5 folds, plus YOLO11s at
  1536 px and YOLO11m at 1280 px on the fold-0 split (extra voters). 40
  epochs, seed 42, early stopping after 10 flat epochs; labels are the union
  of all records' boxes.
* 5 refiners: U-Net with a ResNet-34 encoder on 512 x 512 crops of
  `[image, CLAHE, seed box]`, one per fold, 12 epochs, 0.45 BCE + 0.55 Dice.

Set `RUN_TRAINING = True` to train everything here; the same commands are in
the README as shell lines.

In [ ]:
from filament.detector import train_detector
from filament.refiner import train_refiner

DETECTOR_RUNS = [dict(fold=f, out=WEIGHTS / f"det_f{f}.pt") for f in range(5)] + [
    dict(fold=0, out=WEIGHTS / "det_s1536_f0.pt", imgsz=1536),
    dict(fold=0, out=WEIGHTS / "det_m1280_f0.pt", model="yolo11m.pt", batch=6),
]
if RUN_TRAINING:
    for kw in DETECTOR_RUNS:
        train_detector(work=REPO / "runs", **kw)
    for f in range(5):
        train_refiner(f, WEIGHTS / f"refiner_f{f}.pt")
else:
    print("RUN_TRAINING is False; planned runs:")
    for kw in DETECTOR_RUNS:
        print("  detector", kw)
    for f in range(5):
        print("  refiner ", dict(fold=f, out=WEIGHTS / f"refiner_f{f}.pt"))

## 6. Weights

Inference needs the 12 files below in `weights/`. If they are missing and
`WEIGHTS_URL` (in `filament/ensemble.py`) points to the released archive, it
is downloaded and unpacked here.

In [ ]:
import urllib.request
import zipfile

from filament.ensemble import DET_FILES, REFINER_FILES, WEIGHTS_URL


def missing_weights(names=DET_FILES + REFINER_FILES):
    return [n for n in names if not (WEIGHTS / n).exists()]


if missing_weights() and "REPLACE-ME" not in WEIGHTS_URL:
    WEIGHTS.mkdir(parents=True, exist_ok=True)
    archive = WEIGHTS / "weights.zip"
    urllib.request.urlretrieve(WEIGHTS_URL, archive)
    with zipfile.ZipFile(archive) as z:
        z.extractall(WEIGHTS)
    archive.unlink()
print("missing weights:", missing_weights() or "none")

## 7. Inference on the test set

The defaults of `EnsembleConfig` are exactly the submitted configuration. The
command-line equivalent is
`python -m filament.ensemble --weights weights --out submission.csv`.

In [ ]:
from ultralytics import YOLO

from filament.ensemble import EnsembleConfig, predict
from filament.inference import load_refiner
from filament.submission import validate, write_submission

SUBMISSION = REPO / "submission.csv"
if RUN_INFERENCE and not missing_weights():
    cfg = EnsembleConfig()
    print(cfg)
    dets = [YOLO(str(WEIGHTS / n)) for n in DET_FILES]
    refiners = [load_refiner(WEIGHTS / n, DEVICE) for n in REFINER_FILES]
    test_dir = data_root() / TEST_IMAGES
    test_stems = sorted(p.stem for p in test_dir.glob("*.jpeg"))
    rows = predict(dets, refiners, test_dir, test_stems, DEVICE, cfg, paste="nearest")
    write_submission(rows, SUBMISSION)
    print(f"wrote {SUBMISSION}: {len(rows)} instances on {len(test_stems)} images")
else:
    print("skipped (RUN_INFERENCE is False or weights are missing)")

## 8. Submission validation

The host rejects files with overlapping instances; the validator also checks
the format and that every image id is a test image.

In [ ]:
if SUBMISSION.exists():
    print(validate(SUBMISSION, data_root() / TEST_IMAGES))
else:
    print("no submission.csv yet")

## 9. Fold-0 evaluation with the exact metric

The ensemble cannot be cross-validated honestly (the other folds' models were
trained on fold 0's images). What can be measured without leakage is fold 0's
*own* detector and refiner on fold 0's validation images, run through the same
voting, majority and painting code with one voter (`min_votes=1`). Expect a PQ
around 0.4; the historical single-model figure of 0.4151 used a slightly
different operating point (conf >= 0.46, contrast gate before painting, no
flip views).

In [ ]:
from dataclasses import replace

from filament.data import fold_stems

OOF = REPO / "oof_f0.csv"
if RUN_FOLD0_EVAL and not missing_weights(["det_f0.pt", "refiner_f0.pt"]):
    cfg0 = replace(EnsembleConfig(), sizes=(1280,), min_votes=1)
    val_stems = sorted(fold_stems(0))[:FOLD0_IMAGES]
    rows = predict([YOLO(str(WEIGHTS / "det_f0.pt"))], [load_refiner(WEIGHTS / "refiner_f0.pt", DEVICE)],
                   data_root() / TRAIN_IMAGES, val_stems, DEVICE, cfg0)
    write_submission(rows, OOF)  # CLI equivalent: python -m filament.metric --pred oof_f0.csv --fold 0
    gt = gt_rows_for([r for r in records if r.stem in set(val_stems)])
    print(f"fold-0 PQ on {len(val_stems)} images:", pq_from_rles(gt, rows))
else:
    print("skipped (RUN_FOLD0_EVAL is False or fold-0 weights are missing)")

## 10. A look at the predictions

Predicted instances (filled, one colour each) against the annotators'
outlines (white) on a fold-0 image, or on a test image when only the
submission exists.

In [ ]:
import pandas as pd

from filament.submission import rles_to_layers

source = OOF if OOF.exists() else SUBMISSION if SUBMISSION.exists() else None
if source is None:
    print("no predictions yet (run section 7 or 9)")
else:
    pred = pd.read_csv(source)
    pred["stem"] = pred.filament_id.str.rsplit("_", n=1).str[0]
    show = pred.stem.value_counts().index[0]  # the image with the most instances
    img_dir = data_root() / (TRAIN_IMAGES if source == OOF else TEST_IMAGES)
    img = load_image_gray(img_dir, show + ".jpeg")
    layers = rles_to_layers(pred[pred.stem == show].segmentation_rle.tolist())
    label = np.zeros(img.shape, np.int32)
    for k, layer in enumerate(layers, 1):
        label[layer > 0] = k
    ys, xs = np.nonzero(layers[int(np.argmax(layers.sum(axis=(1, 2))))])  # zoom on the largest instance
    x0, y0 = int(np.clip(xs.mean() - 256, 0, 1536)), int(np.clip(ys.mean() - 256, 0, 1536))
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    for ax, (xa, xb, ya, yb) in zip(axes, [(0, 2048, 0, 2048), (x0, x0 + 512, y0, y0 + 512)]):
        ax.imshow(img, cmap="gray")
        ax.imshow(np.ma.masked_equal(label, 0), cmap="tab20", alpha=0.5, interpolation="nearest")
        for rec in by_stem.get(show, []):
            for f in rec.filaments:
                poly = np.array(f.polygon).reshape(-1, 2)
                ax.plot(*np.vstack([poly, poly[:1]]).T, color="white", lw=0.6)
        ax.set_xlim(xa, xb)
        ax.set_ylim(yb, ya)
        ax.axis("off")
    axes[0].set_title(f"{show}: {len(layers)} predicted instances ({source.name})")
    axes[1].set_title("512 x 512 px zoom")
    plt.tight_layout()
    plt.show()

## Notes

* Every setting above is a default of the package CLIs; the README lists the
  equivalent shell commands.
* `tests/test_equivalence.py` replays this code against the original research
  code on real checkpoints and asserts identical masks.
* Only the competition's training data and the test H-alpha images are used.